In [1]:
from pathlib import Path
from pypdf import PdfReader
import pandas as pd

print("Entorno listo")

Entorno listo


In [2]:
from pathlib import Path

ruta_pdf = Path("../data/raw/pdf_estudiantes")

archivos = list(ruta_pdf.glob("*.pdf"))

print("Cantidad de PDFs encontrados:", len(archivos))

for archivo in archivos:
    print(archivo.name)

Cantidad de PDFs encontrados: 5
45506034.pdf
60108703.pdf
60910993.pdf
72089261.pdf
72611344.pdf


In [3]:
from pypdf import PdfReader

pdf_prueba = archivos[0]

reader = PdfReader(pdf_prueba)

texto = ""

for pagina in reader.pages:
    texto += pagina.extract_text()

print(texto[:3000])

FICHA DE RENDIMIENTO ACADÉMICO
NOMBRE DE LA INSTITUCIÓN INSTITUTO DE EDUCACIÓN SUPERIOR PEDAGÓGICO PÚBLICO "GUSTAVO ALLENDE LLAVERÍA " - TARMA
PROGRAMA DE ESTUDIOS EDUCACIÓN FÍSICA (R.V.M N° 147 - 2020 - MINEDU)
APELLIDOS Y NOMBRES AGUILAR JIMENEZ ELEAZAR ELVIS MODALIDAD PRESENCIAL
CÓDIGO / MATRÍCULA 45506034 DNI 45506034
Periodo Académico Ciclo
2020-II I
N° CURSOS CALIFICACIÓN
CUALITATIVA NOTA CRED. PUNTAJE OBSERVACIÓN
1 DESARROLLO PERSONAL I EN PROCESO 13 5 65
2 EDUCACIÓN Y SOCIEDAD EN EL SIGLO XXI EN PROCESO 11 5 55
3 HISTORIA, SOCIEDAD Y DIVERSIDAD EN PROCESO 13 3 39
4 LECTURA Y ESCRITURA EN LA EDUCACIÓN SUPERIOR LOGRADO 15 3 45
5 PRÁCTICA E INVESTIGACIÓN I INICIO 9 5 45
6 RESOLUCIÓN DE PROBLEMAS MATEMÁTICOS I EN PROCESO 13 3 39
PROMEDIO DEL PERIODO 12.00
Periodo Académico Ciclo
2021-I II
N° CURSOS CALIFICACIÓN
CUALITATIVA NOTA CRED. PUNTAJE OBSERVACIÓN
1 PRÁCTICA E INVESTIGACIÓN I PREVIO AL
INICIO 0 5 0
2 CIENCIA Y EPISTEMOLOGÍAS PREVIO AL
INICIO 0 3 0
3 COMUNICACIÓN ORAL EN LA ED

In [4]:
import re

periodos = re.findall(
    r'Periodo Académico\s+Ciclo\s+(\d{4}-[I|V]+)\s+([IVX]+)',
    texto
)

periodos

[('2020-II', 'I'),
 ('2021-I', 'II'),
 ('2021-II', 'II'),
 ('2022-I', 'III'),
 ('2022-II', 'IV'),
 ('2023-I', 'V'),
 ('2023-II', 'VI'),
 ('2024-I', 'VII'),
 ('2024-II', 'VIII'),
 ('2025-I', 'IX'),
 ('2025-II', 'X')]

In [7]:
import re

patron_bloque = r'Periodo Académico\s+Ciclo\s+(\d{4}-[IV]+)\s+([IVX]+)(.*?)(?=Periodo Académico|$)'

periodos = re.findall(patron_bloque, texto, re.S)

print("Cantidad de periodos encontrados:", len(periodos))

for periodo, ciclo, contenido in periodos[:2]:
    print("PERIODO:", periodo)
    print("CICLO:", ciclo)
    print(contenido[:500])
    print("----------------")

Cantidad de periodos encontrados: 11
PERIODO: 2020-II
CICLO: I

N° CURSOS CALIFICACIÓN
CUALITATIVA NOTA CRED. PUNTAJE OBSERVACIÓN
1 DESARROLLO PERSONAL I EN PROCESO 13 5 65
2 EDUCACIÓN Y SOCIEDAD EN EL SIGLO XXI EN PROCESO 11 5 55
3 HISTORIA, SOCIEDAD Y DIVERSIDAD EN PROCESO 13 3 39
4 LECTURA Y ESCRITURA EN LA EDUCACIÓN SUPERIOR LOGRADO 15 3 45
5 PRÁCTICA E INVESTIGACIÓN I INICIO 9 5 45
6 RESOLUCIÓN DE PROBLEMAS MATEMÁTICOS I EN PROCESO 13 3 39
PROMEDIO DEL PERIODO 12.00

----------------
PERIODO: 2021-I
CICLO: II

N° CURSOS CALIFICACIÓN
CUALITATIVA NOTA CRED. PUNTAJE OBSERVACIÓN
1 PRÁCTICA E INVESTIGACIÓN I PREVIO AL
INICIO 0 5 0
2 CIENCIA Y EPISTEMOLOGÍAS PREVIO AL
INICIO 0 3 0
3 COMUNICACIÓN ORAL EN LA EDUCACIÓN SUPERIOR PREVIO AL
INICIO 0 3 0
4 NIÑEZ Y ADOLESCENCIAS DESARROLLO CAMBIOS E IDENTIDADES PREVIO AL
INICIO 0 5 0
5 PLANIFICACIÓN MEDIACIÓN Y EVALUACIÓN DE LOS
APRENDIZAJES I
PREVIO AL
INICIO 0 5 0
6 PRÁCTICA E INVESTIGACIÓN II PREVIO AL
INICIO 0 5 0
7 RESOLUCIÓN DE PROBLEMAS 

In [23]:
def extraer_cursos(bloque, periodo, ciclo):
    
    cursos = []

    # Unir saltos de línea
    bloque = bloque.replace("\n", " ")

    # Limpiar espacios dobles
    bloque = re.sub(r'\s+', ' ', bloque)

    # Capturar curso + estado + nota + crédito + puntaje
    patron = r'\d+\s+(.*?)\s+(PREVIO AL INICIO|LOGRADO|EN PROCESO|INICIO)\s+(\d+)\s+(\d+)\s+(\d+)'

    resultados = re.findall(patron, bloque)

    for curso, estado, nota, credito, puntaje in resultados:

        cursos.append({
            "periodo": periodo,
            "ciclo": ciclo,
            "curso": curso.strip(),
            "estado": estado.strip(),
            "nota": int(nota),
            "credito": int(credito),
            "puntaje": int(puntaje)
        })

    return cursos

In [9]:
primer_periodo = periodos[0]

datos_cursos = extraer_cursos(
    primer_periodo[2],
    primer_periodo[0],
    primer_periodo[1]
)

datos_cursos

[{'periodo': '2020-II',
  'ciclo': 'I',
  'curso': 'DESARROLLO PERSONAL I',
  'estado': 'EN PROCESO',
  'nota': 13,
  'credito': 5,
  'puntaje': 65},
 {'periodo': '2020-II',
  'ciclo': 'I',
  'curso': 'EDUCACIÓN Y SOCIEDAD EN EL SIGLO XXI',
  'estado': 'EN PROCESO',
  'nota': 11,
  'credito': 5,
  'puntaje': 55},
 {'periodo': '2020-II',
  'ciclo': 'I',
  'curso': 'HISTORIA, SOCIEDAD Y DIVERSIDAD',
  'estado': 'EN PROCESO',
  'nota': 13,
  'credito': 3,
  'puntaje': 39},
 {'periodo': '2020-II',
  'ciclo': 'I',
  'curso': 'LECTURA Y ESCRITURA EN LA EDUCACIÓN SUPERIOR',
  'estado': 'LOGRADO',
  'nota': 15,
  'credito': 3,
  'puntaje': 45},
 {'periodo': '2020-II',
  'ciclo': 'I',
  'curso': 'PRÁCTICA E INVESTIGACIÓN I',
  'estado': 'INICIO',
  'nota': 9,
  'credito': 5,
  'puntaje': 45},
 {'periodo': '2020-II',
  'ciclo': 'I',
  'curso': 'RESOLUCIÓN DE PROBLEMAS MATEMÁTICOS I',
  'estado': 'EN PROCESO',
  'nota': 13,
  'credito': 3,
  'puntaje': 39}]

In [10]:
def extraer_id_estudiante(texto):
    patron = r'CÓDIGO / MATRÍCULA\s+(\d+)'
    
    resultado = re.search(patron, texto)
    
    if resultado:
        return resultado.group(1)
    
    return None

In [11]:
id_estudiante = extraer_id_estudiante(texto)

print(id_estudiante)

45506034


In [24]:
def procesar_pdf(ruta_pdf):
    
    # Leer PDF
    reader = PdfReader(ruta_pdf)

    texto = ""

    for pagina in reader.pages:
        texto += pagina.extract_text()

    # Obtener estudiante
    id_estudiante = extraer_id_estudiante(texto)

    # Obtener periodos
    patron_bloque = r'Periodo Académico\s+Ciclo\s+(\d{4}-[IV]+)\s+([IVX]+)(.*?)(?=Periodo Académico|$)'

    periodos = re.findall(patron_bloque, texto, re.S)

    registros = []

    # Extraer cursos de cada periodo
    for periodo, ciclo, contenido in periodos:

        cursos = extraer_cursos(
            contenido,
            periodo,
            ciclo
        )

        for curso in cursos:
            curso["id_estudiante"] = id_estudiante
            registros.append(curso)

    return registros

In [13]:
ruta_prueba = archivos[0]

registros = procesar_pdf(ruta_prueba)

len(registros)

64

In [14]:
registros[:5]

[{'periodo': '2020-II',
  'ciclo': 'I',
  'curso': 'DESARROLLO PERSONAL I',
  'estado': 'EN PROCESO',
  'nota': 13,
  'credito': 5,
  'puntaje': 65,
  'id_estudiante': '45506034'},
 {'periodo': '2020-II',
  'ciclo': 'I',
  'curso': 'EDUCACIÓN Y SOCIEDAD EN EL SIGLO XXI',
  'estado': 'EN PROCESO',
  'nota': 11,
  'credito': 5,
  'puntaje': 55,
  'id_estudiante': '45506034'},
 {'periodo': '2020-II',
  'ciclo': 'I',
  'curso': 'HISTORIA, SOCIEDAD Y DIVERSIDAD',
  'estado': 'EN PROCESO',
  'nota': 13,
  'credito': 3,
  'puntaje': 39,
  'id_estudiante': '45506034'},
 {'periodo': '2020-II',
  'ciclo': 'I',
  'curso': 'LECTURA Y ESCRITURA EN LA EDUCACIÓN SUPERIOR',
  'estado': 'LOGRADO',
  'nota': 15,
  'credito': 3,
  'puntaje': 45,
  'id_estudiante': '45506034'},
 {'periodo': '2020-II',
  'ciclo': 'I',
  'curso': 'PRÁCTICA E INVESTIGACIÓN I',
  'estado': 'INICIO',
  'nota': 9,
  'credito': 5,
  'puntaje': 45,
  'id_estudiante': '45506034'}]

In [25]:
todos_los_registros = []

for archivo in archivos:
    
    print("Procesando:", archivo.name)
    
    registros = procesar_pdf(archivo)
    
    todos_los_registros.extend(registros)

print("Total de registros extraídos:", len(todos_los_registros))

Procesando: 45506034.pdf
Procesando: 60108703.pdf
Procesando: 60910993.pdf
Procesando: 72089261.pdf
Procesando: 72611344.pdf
Total de registros extraídos: 263


In [26]:
df = pd.DataFrame(todos_los_registros)

df.head()

,periodo,ciclo,curso,estado,nota,credito,puntaje,id_estudiante
0,2020-II,I,DESARROLLO PERSONAL I,EN PROCESO,13,5,65,45506034
1,2020-II,I,EDUCACIÓN Y SOCIEDAD EN EL SIGLO XXI,EN PROCESO,11,5,55,45506034
2,2020-II,I,"HISTORIA, SOCIEDAD Y DIVERSIDAD",EN PROCESO,13,3,39,45506034
3,2020-II,I,LECTURA Y ESCRITURA EN LA EDUCACIÓN SUPERIOR,LOGRADO,15,3,45,45506034
4,2020-II,I,PRÁCTICA E INVESTIGACIÓN I,INICIO,9,5,45,45506034


In [33]:
df.to_csv(
    "../data/interim/registros_academicos_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV generado correctamente")

CSV generado correctamente


In [18]:
df[df["estado"] == "PREVIO AL"].head()

,periodo,ciclo,curso,estado,nota,credito,puntaje,id_estudiante


In [19]:
df.shape

(263, 8)

In [28]:
df[df["curso"].str.contains("PLANIFICACIÓN", case=False, na=False)]

,periodo,ciclo,curso,estado,nota,credito,puntaje,id_estudiante
10,2021-I,II,PLANIFICACIÓN MEDIACIÓN Y EVALUACIÓN DE LOS AP...,PREVIO AL INICIO,0,5,0,45506034
17,2021-II,II,PLANIFICACIÓN MEDIACIÓN Y EVALUACIÓN DE LOS AP...,EN PROCESO,14,5,70,45506034
32,2022-II,IV,PLANIFICACIÓN MEDIACIÓN Y EVALUACIÓN DE LOS AP...,EN PROCESO,13,3,39,45506034
47,2023-II,VI,PLANIFICACIÓN MEDIACIÓN Y EVALUACIÓN DE LOS AP...,EN PROCESO,13,3,39,45506034
73,2026-I,II,PLANIFICACIÓN POR COMPETENCIAS Y EVALUACIÓN PA...,LOGRADO,15,5,75,60108703
85,2024-II,II,PLANIFICACIÓN POR COMPETENCIAS Y EVALUACIÓN PA...,EN PROCESO,13,5,65,60910993
100,2025-II,IV,PLANIFICACIÓN POR COMPETENCIAS Y EVALUACIÓN PA...,EN PROCESO,14,3,42,60910993
155,2022-I,II,PLANIFICACIÓN POR COMPETENCIAS Y EVALUACIÓN PA...,INICIO,9,5,45,72089261
158,2022-II,III,PLANIFICACIÓN POR COMPETENCIAS Y EVALUACIÓN PA...,EN PROCESO,14,5,70,72089261
171,2023-I,IV,PLANIFICACIÓN POR COMPETENCIAS Y EVALUACIÓN PA...,INICIO,7,3,21,72089261


In [27]:
df["estado"].value_counts()

estado
EN PROCESO          152
LOGRADO              88
INICIO               12
PREVIO AL INICIO     11
Name: count, dtype: int64

In [29]:
df[df["curso"].str.contains("\n", na=False)]

,periodo,ciclo,curso,estado,nota,credito,puntaje,id_estudiante


In [30]:
df["id_estudiante"].nunique()

5

In [31]:
df.groupby("id_estudiante").size()

id_estudiante
45506034    64
60108703    12
60910993    33
72089261    64
72611344    90
dtype: int64

In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 263 entries, 0 to 262
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   periodo        263 non-null    str  
 1   ciclo          263 non-null    str  
 2   curso          263 non-null    str  
 3   estado         263 non-null    str  
 4   nota           263 non-null    int64
 5   credito        263 non-null    int64
 6   puntaje        263 non-null    int64
 7   id_estudiante  263 non-null    str  
dtypes: int64(3), str(5)
memory usage: 16.6 KB
